# Transforming Data

The explainer introduced tidy data principles and four transformations — filtering, deriving, reshaping, and aggregating — that turn raw data into something ready for analysis. We saw that the shape data arrives in is almost never the shape we need, and that recognising what transformation is required is as important as knowing the syntax.

This notebook applies those ideas to the logistics scenario. The COO needs a cost analysis by region and carrier, along with summary tables that can go straight into a board presentation. The combined shipments data is loaded and clean. Now it needs reshaping: new derived columns, categorisation into bins, grouped aggregations, and pivoted summaries.

By the end of this notebook we will be able to:

- Create new columns from existing ones (derived columns)
- Categorise continuous values into bins with `pd.cut`
- Apply custom functions with `.apply()` and `.map()`
- Aggregate data with `.groupby().agg()` using multiple functions
- Reshape data with `pd.pivot_table`
- Convert between wide and long formats with `.melt()` and `.pivot()`

In [ ]:
%pip install -q pandas

In [ ]:
import pandas as pd
import json
import sqlite3

### Loading and preparing the data

We repeat the loading and cleaning pipeline from the previous notebooks in a compact form. This is the extraction and cleaning work we have already done — compressed into a single cell so we can focus on transformation.

In [ ]:
# --- Load Asia ---
asia = pd.read_csv("../data/shipments_asia.csv")
asia["region"] = "Asia"

# --- Load Europe (fix column names and dates) ---
europe = pd.read_csv("../data/shipments_europe.csv")
europe = europe.rename(columns={"id": "shipment_id", "cost_eur": "cost_local"})
europe["ship_date"] = pd.to_datetime(europe["ship_date"], format="%d/%m/%Y").dt.strftime("%Y-%m-%d")
europe["cost_usd"] = (europe["cost_local"] * 1.08).round(2)
europe = europe.drop(columns=["cost_local"])
europe["region"] = "Europe"

# --- Load Americas (flatten JSON) ---
with open("../data/shipments_americas.json") as f:
    americas_raw = json.load(f)
americas = pd.json_normalize(americas_raw)
americas = americas.rename(columns={
    "route.origin": "origin",
    "route.destination": "destination",
    "details.weight_kg": "weight_kg",
    "details.ship_date_unix": "ship_date_unix",
    "financials.cost_usd": "cost_usd",
    "financials.insurance_usd": "insurance_usd",
})
americas["ship_date"] = pd.to_datetime(americas["ship_date_unix"], unit="s").dt.strftime("%Y-%m-%d")
americas = americas.drop(columns=["ship_date_unix"])
americas["region"] = "Americas"

# --- Concatenate ---
shipments = pd.concat([asia, europe, americas], ignore_index=True)
shipments["ship_date"] = pd.to_datetime(shipments["ship_date"])

print(f"Shape: {shipments.shape}")
shipments.head(3)

---

## 1. Creating derived columns

The explainer described derived columns as new information calculated from existing ones: durations from start and end dates, totals from quantity and price. The raw data contains the building blocks, but not the specific metrics the analysis needs. Deriving columns bridges that gap.

In [ ]:
# Cost per kilogram
shipments["cost_per_kg"] = (shipments["cost_usd"] / shipments["weight_kg"]).round(2)

# Month of shipment (extracted from the datetime column)
shipments["ship_month"] = shipments["ship_date"].dt.month

# Quarter
shipments["ship_quarter"] = shipments["ship_date"].dt.quarter

# Flag: is the shipment delayed or in customs hold?
shipments["is_problem"] = shipments["status"].isin(["delayed", "customs_hold"])

shipments[["shipment_id", "cost_usd", "weight_kg", "cost_per_kg", "ship_month", "ship_quarter", "is_problem"]].head()

---

## 2. Binning with `pd.cut`

Sometimes we need to turn a continuous variable into categories. The COO does not want to see 150 individual weight values — they want to know how many shipments are parcels, how many are light freight, and how many are heavy. **`pd.cut`** divides a continuous variable into discrete bins (categories). We provide the column, the bin edges, and optionally the labels.

In [ ]:
# Weight categories
weight_bins = [0, 10, 100, 1000, 5000]
weight_labels = ["Parcel (<10kg)", "Light (10-100kg)", "Medium (100-1000kg)", "Heavy (1000-5000kg)"]

shipments["weight_category"] = pd.cut(
    shipments["weight_kg"],
    bins=weight_bins,
    labels=weight_labels,
    include_lowest=True,
)

print(shipments["weight_category"].value_counts().sort_index())

In [ ]:
# Cost brackets
cost_bins = [0, 1000, 5000, 15000, 50000]
cost_labels = ["Budget (<$1k)", "Standard ($1k-$5k)", "Premium ($5k-$15k)", "High-value ($15k+)"]

shipments["cost_bracket"] = pd.cut(
    shipments["cost_usd"],
    bins=cost_bins,
    labels=cost_labels,
    include_lowest=True,
)

print(shipments["cost_bracket"].value_counts().sort_index())

---

## 3. `.apply()` and `.map()`

Not every transformation fits neatly into arithmetic or binning. Sometimes we need to map values through a lookup table, or apply a rule that involves multiple columns. pandas gives us two tools for this.

**`.map()`** replaces values in a Series using a dictionary or function. It works element by element.

**`.apply()`** runs a function on each element of a Series (or each row/column of a DataFrame). It is more flexible than `.map()` but typically slower for simple operations.

In [ ]:
# .map() with a dictionary: convert status codes to human-readable labels
status_labels = {
    "delivered": "Delivered",
    "in_transit": "In Transit",
    "delayed": "Delayed",
    "customs_hold": "Customs Hold",
}

shipments["status_label"] = shipments["status"].map(status_labels)
shipments[["status", "status_label"]].drop_duplicates()

In [ ]:
# .apply() with a custom function: classify cost efficiency
def classify_efficiency(row):
    """Classify a shipment's cost efficiency based on cost per kg."""
    cpk = row["cost_per_kg"]
    if cpk < 5:
        return "efficient"
    elif cpk < 20:
        return "moderate"
    else:
        return "expensive"

shipments["efficiency"] = shipments.apply(classify_efficiency, axis=1)
print(shipments["efficiency"].value_counts())

Note: `.apply(axis=1)` passes each **row** as a Series to the function. With `axis=0` (the default), it passes each column. For row-level logic, always use `axis=1`.

When the logic is simple, vectorised operations (like `pd.cut` or boolean indexing) are faster than `.apply()`. Use `.apply()` when the logic is too complex for a single expression.

---

## 4. Grouped aggregation with `.groupby().agg()`

The explainer described aggregation as collapsing many rows into fewer rows by grouping and summarising. This is the "for each X, what is the Y?" pattern we have seen in SQL. In pandas, **`.groupby()`** splits a DataFrame into groups based on one or more columns, and **`.agg()`** applies one or more aggregation functions to each group.

This is where the COO's questions start getting answered directly.

In [ ]:
# Summary by region
region_summary = shipments.groupby("region").agg(
    shipment_count=("shipment_id", "count"),
    total_cost=("cost_usd", "sum"),
    avg_cost=("cost_usd", "mean"),
    avg_weight=("weight_kg", "mean"),
    problem_rate=("is_problem", "mean"),
).round(2)

region_summary

In [ ]:
# Summary by region and carrier
carrier_summary = (
    shipments
    .groupby(["region", "carrier"])
    .agg(
        count=("shipment_id", "count"),
        total_cost=("cost_usd", "sum"),
        avg_cost_per_kg=("cost_per_kg", "mean"),
    )
    .round(2)
    .sort_values("total_cost", ascending=False)
)

carrier_summary.head(10)

The named aggregation syntax `column_name=("source_column", "function")` is the clearest way to define multiple aggregations. Each tuple specifies which column to aggregate and which function to use.

---

## 5. Pivot tables with `pd.pivot_table`

The explainer discussed the difference between wide and long format. Aggregated data in long format is useful for further analysis, but the COO wants a summary table for a board presentation — and board presentations need data in wide format, arranged as a readable grid.

A **pivot table** rearranges data into that matrix format. One column's values become the new column headers, another becomes the row index, and a third is aggregated in the cells. This is the pandas equivalent of a pivot table in a spreadsheet.

In [ ]:
# Average cost by region and status
cost_pivot = pd.pivot_table(
    shipments,
    values="cost_usd",
    index="region",
    columns="status",
    aggfunc="mean",
).round(2)

cost_pivot

In [ ]:
# Shipment count by region and quarter
count_pivot = pd.pivot_table(
    shipments,
    values="shipment_id",
    index="region",
    columns="ship_quarter",
    aggfunc="count",
    fill_value=0,
)
count_pivot.columns = [f"Q{q}" for q in count_pivot.columns]

count_pivot

---

## 6. Wide vs long format: `.melt()` and `.pivot()`

The explainer described this as the central tension of data structure: analytical tools work best with long format (each observation in its own row), but humans read wide format more easily (categories spread across columns). As analysts, we will frequently convert between the two.

- **Wide format**: each category gets its own column (like the pivot tables above)
- **Long format**: categories are stored as values in a single column (one row per observation)

**`.melt()`** converts from wide to long. **`.pivot()`** converts from long to wide.

In [ ]:
# The count_pivot is in wide format. Convert to long with .melt()
count_long = count_pivot.reset_index().melt(
    id_vars="region",
    var_name="quarter",
    value_name="shipment_count",
)

print("Long format:")
count_long

In [ ]:
# Convert back to wide format with .pivot()
count_wide = count_long.pivot(
    index="region",
    columns="quarter",
    values="shipment_count",
)

print("Wide format (round-trip):")
count_wide

The key difference between `.pivot()` and `pd.pivot_table()`: `.pivot()` simply rearranges existing values (no aggregation), while `pd.pivot_table()` can aggregate (sum, mean, count, etc.) when there are multiple values per cell.

---

## Exercises

Now it is your turn. These exercises build towards the kind of output the COO actually needs.

Complete each exercise in the code cell provided.

### Exercise 1: Derived columns and binning

Create the following new columns on the `shipments` DataFrame:

1. `cost_band`: use `pd.cut` to bin `cost_usd` into 5 equal-width bins (use `pd.cut(shipments["cost_usd"], bins=5)`)
2. `ship_day_of_week`: the day of the week name (Monday, Tuesday, etc.) extracted from `ship_date` using `.dt.day_name()`
3. `is_heavy`: a boolean column that is `True` when `weight_kg` exceeds 1000

Print the value counts for each new column.

In [ ]:
# Your code here


### Exercise 2: Grouped aggregation

Using `.groupby().agg()`, create a summary table grouped by `region` and `weight_category` with the following columns:

- `count`: number of shipments
- `total_weight_kg`: sum of `weight_kg`
- `mean_cost_usd`: mean of `cost_usd`
- `max_cost_per_kg`: maximum of `cost_per_kg`

Sort the result by `total_weight_kg` descending. Which region and weight category combination accounts for the most total weight?

In [ ]:
# Your code here


### Exercise 3: Pivot table for the COO

Create a pivot table that shows the **total cost in USD** by `region` (rows) and `ship_quarter` (columns). Add a column for the row total using `.assign(total=lambda df: df.sum(axis=1))`. Format the numbers so they are readable (round to the nearest integer).

This is the kind of table that goes into an executive summary.

In [ ]:
# Your code here


### Exercise 4: Melt and reshape

Take the pivot table you created in Exercise 3 (without the total column) and:

1. Use `.melt()` to convert it to long format with columns `region`, `quarter`, and `total_cost`
2. Filter to only Q1 and Q4
3. Use `.pivot()` to reshape it so that `region` is the index and Q1 and Q4 are separate columns
4. Add a column `q4_vs_q1_change` that shows the difference between Q4 and Q1 costs

Print the result. Which region had the largest absolute change between Q1 and Q4?

In [ ]:
# Your code here


---

## Summary

We started with clean, combined shipment data and transformed it into something that can directly answer the COO's questions: cost breakdowns by region and carrier, quarterly trends, and presentation-ready summary tables.

In this notebook we:

- Created **derived columns** from arithmetic operations and date extraction
- Used **`pd.cut`** to bin continuous values into labelled categories
- Mapped values with **`.map()`** and applied row-level logic with **`.apply()`**
- Aggregated data with **`.groupby().agg()`** using named aggregation tuples
- Built summary matrices with **`pd.pivot_table`**
- Converted between wide and long formats with **`.melt()`** and **`.pivot()`**

Together with the merging and cleaning techniques from the previous notebooks, we now have the full pipeline: extracting data from different sources, cleaning it, and transforming it into the shape our analysis requires. That is the ETL process the explainer introduced, done in code rather than by hand.